In [16]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
import os

In [17]:
load_dotenv()

True

In [18]:
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
    api_key=os.getenv('GROQ_API_KEY')
)

In [19]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [20]:
def chat(state:ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {
        'messages': [response]
    }


In [21]:
checkpoint = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat', chat)

graph.add_edge(START, 'chat')
graph.add_edge('chat', END)

chatbot = graph.compile(checkpointer = checkpoint)


In [23]:
thread_id = "1"
while True:
    user_message = input('Type here:')
    print('User: ', user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]},config=config)
    print('AI: ', response['messages'][-1].content)

User:  hi, my name is juan
AI:  Hello Juan! 👋 How can I help you today?
User:  do you remember my name?
AI:  Yes, you mentioned that your name is Juan. If there’s anything else you’d like to share or ask, feel free!
User:  bye
